In [ ]:
import os
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Configura tu ruta ──────────────────────────────────────────────────────────
soccernet_path = "/path/to/soccernet/calibration-2023"   # <-- cambia esto

img_sample   = os.path.join(soccernet_path, "train", "03207.jpg")
label_sample = img_sample.replace(".jpg", ".json")
# ──────────────────────────────────────────────────────────────────────────────

# Paleta de colores por categoría (BGR para OpenCV)
LINE_COLORS = {
    # Área grande
    "Big rect. left bottom":   (0,   200, 255),
    "Big rect. left main":     (0,   200, 255),
    "Big rect. left top":      (0,   200, 255),
    "Big rect. right bottom":  (0,   140, 255),
    "Big rect. right main":    (0,   140, 255),
    "Big rect. right top":     (0,   140, 255),
    # Área pequeña
    "Small rect. left bottom": (255, 180,   0),
    "Small rect. left main":   (255, 180,   0),
    "Small rect. left top":    (255, 180,   0),
    "Small rect. right bottom":(255, 120,   0),
    "Small rect. right main":  (255, 120,   0),
    "Small rect. right top":   (255, 120,   0),
    # Líneas de campo
    "Side line top":           (100, 255, 100),
    "Side line bottom":        (100, 255, 100),
    "Side line left":          ( 50, 200,  50),
    "Side line right":         ( 50, 200,  50),
    "Middle line":             (255, 255,   0),
    # Círculo central
    "Circle central":          (200,   0, 200),
    "Circle left":             (180,   0, 180),
    "Circle right":            (180,   0, 180),
    # Porterías
    "Goal left crossbar":      (  0,   0, 255),
    "Goal right crossbar":     (  0,   0, 255),
    "Goal left post left ":    (  0,  60, 255),
    "Goal left post right":    (  0,  60, 255),
    "Goal right post left":    (  0,  60, 255),
    "Goal right post right ":  (  0,  60, 255),
}
DEFAULT_COLOR = (200, 200, 200)


def draw_annotations(img_path, label_path, thickness=2):
    img = cv2.imread(img_path)
    if img is None:
        raise FileNotFoundError(f"No se pudo leer la imagen: {img_path}")

    with open(label_path) as f:
        labels = json.load(f)

    drawn_labels = {}
    for line_name, data in labels.items():
        color = LINE_COLORS.get(line_name, DEFAULT_COLOR)
        segments = data.get("lines", [])
        for seg in segments:
            if "x1" in seg:
                pt1 = (int(round(seg["x1"])), int(round(seg["y1"])))
                pt2 = (int(round(seg["x2"])), int(round(seg["y2"])))
            elif "points" in seg:
                pts = seg["points"]
                pt1 = (int(round(pts[0]["x"])),  int(round(pts[0]["y"])))
                pt2 = (int(round(pts[-1]["x"])), int(round(pts[-1]["y"])))
            else:
                continue
            cv2.line(img, pt1, pt2, color, thickness, cv2.LINE_AA)
            cv2.circle(img, pt1, 4, color, -1)
            cv2.circle(img, pt2, 4, color, -1)
        drawn_labels[line_name] = color
    return img, drawn_labels


# --- Ejecutar y mostrar ---
annotated_img, labels_found = draw_annotations(img_sample, label_sample)

img_rgb = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)
patches = [
    mpatches.Patch(color=tuple(c/255 for c in (r, g, b)), label=name)
    for name, (b, g, r) in labels_found.items()
]

fig, ax = plt.subplots(figsize=(16, 9))
ax.imshow(img_rgb)
ax.axis("off")
ax.set_title("SoccerNet sn-calibration — anotaciones de líneas", fontsize=13)
ax.legend(handles=patches, loc="upper right", fontsize=6.5, framealpha=0.7, ncol=2)
plt.tight_layout()
plt.show()

print(f"\nLíneas encontradas ({len(labels_found)}):")
for name in labels_found:
    print(f"  · {name}")